# 01. EDA — 태양광 패널 이미지 데이터 이해

이 노트북은 태양광 패널 이미지 데이터의 기본 구조와 품질을 확인하기 위한 EDA(탐색적 데이터 분석) 노트북입니다.

## 목표
- 학습/테스트 데이터 구조 확인
- 라벨 분포 확인
- 이미지 샘플 시각화
- 이미지 크기와 비율 확인
- 밝기/색상/선명도 분포 확인
- 중복·라벨 오류·비관련 이미지 가능성 탐색
- 이후 데이터 정제 실험을 위한 기준 만들기

> 이 단계에서는 문제 이미지를 확정하지 않고, 검토가 필요한 후보만 찾습니다.

## 1. 라이브러리 및 기본 설정

In [ ]:
import os
import random
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

warnings.filterwarnings("ignore")

SEED = 42
BASE_PATH = Path("rs18")
TRAIN_DIR = BASE_PATH / "train"
TEST_DIR = BASE_PATH / "test"
LABEL_PATH = BASE_PATH / "train_labels.csv"
SUBMISSION_PATH = BASE_PATH / "sample_submission.csv"

random.seed(SEED)
np.random.seed(SEED)

print("기본 설정 완료")

## 2. 데이터 파일 구조 확인

In [ ]:
print("BASE_PATH:", BASE_PATH.resolve())
print("train 폴더 존재:", TRAIN_DIR.exists())
print("test 폴더 존재:", TEST_DIR.exists())
print("train_labels.csv 존재:", LABEL_PATH.exists())
print("sample_submission.csv 존재:", SUBMISSION_PATH.exists())

print("\n=== 폴더 내부 파일 ===")
if BASE_PATH.exists():
    for item in sorted(BASE_PATH.iterdir()):
        print("-", item.name)

## 3. CSV 구조 및 라벨 분포 확인

In [ ]:
df = pd.read_csv(LABEL_PATH)
sub = pd.read_csv(SUBMISSION_PATH)

print("=== train_labels.csv ===")
print("shape:", df.shape)
display(df.head())
print("컬럼:", df.columns.tolist())

print("\n라벨 분포")
label_counts = df["label"].value_counts().sort_index()
display(label_counts.rename("count").to_frame())

print("\n라벨 비율")
display((label_counts / len(df) * 100).round(2).rename("percent").to_frame())

print("\n=== sample_submission.csv ===")
print("shape:", sub.shape)
display(sub.head())
print("컬럼:", sub.columns.tolist())

## 4. 이미지 파일 개수 및 CSV 일치 여부 확인

In [ ]:
VALID_EXT = {".jpg", ".jpeg", ".png", ".webp"}

train_files = sorted([
    p.name for p in TRAIN_DIR.iterdir()
    if p.suffix.lower() in VALID_EXT
])

test_files = sorted([
    p.name for p in TEST_DIR.iterdir()
    if p.suffix.lower() in VALID_EXT
])

print("train 이미지 수:", len(train_files))
print("test 이미지 수:", len(test_files))
print("train_labels 행 수:", len(df))
print("sample_submission 행 수:", len(sub))

train_ids_from_files = {Path(f).stem for f in train_files}
train_ids_from_csv = set(df["id"])

missing_images = sorted(train_ids_from_csv - train_ids_from_files)
unmatched_files = sorted(train_ids_from_files - train_ids_from_csv)

print("\nCSV에는 있지만 이미지가 없는 ID 수:", len(missing_images))
print("이미지는 있지만 CSV에 없는 ID 수:", len(unmatched_files))

## 5. 라벨별 랜덤 샘플 이미지 확인

In [ ]:
def find_image_path(image_id, directory=TRAIN_DIR):
    for ext in VALID_EXT:
        path = directory / f"{image_id}{ext}"
        if path.exists():
            return path
    raise FileNotFoundError(f"{image_id} 이미지를 찾을 수 없습니다.")

def show_label_samples(dataframe, label, n=20, seed=42):
    sample_df = dataframe[dataframe["label"] == label].sample(
        min(n, (dataframe["label"] == label).sum()),
        random_state=seed
    )

    cols = 5
    rows = int(np.ceil(len(sample_df) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(16, 3.2 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, sample_df.iterrows()):
        img = Image.open(find_image_path(row["id"])).convert("RGB")
        ax.imshow(img)
        ax.set_title(f'{row["id"]}\nlabel={row["label"]}', fontsize=9)
        ax.axis("off")

    for ax in axes[len(sample_df):]:
        ax.axis("off")

    plt.suptitle(f"Random Samples — label {label}", fontsize=16)
    plt.tight_layout()
    plt.show()

show_label_samples(df, label=0, n=20, seed=SEED)
show_label_samples(df, label=1, n=20, seed=SEED)

### 관찰 기록
- 라벨 0에서 눈에 띄는 공통점:
- 라벨 1에서 눈에 띄는 공통점:
- 라벨 기준이 애매해 보이는 이미지:
- 그래픽/제품 이미지 등 off-topic 후보:
- 동일하거나 비슷해 보이는 이미지:

## 6. 이미지 크기와 가로세로 비율 확인

In [ ]:
records = []

for filename in train_files:
    path = TRAIN_DIR / filename
    with Image.open(path) as img:
        width, height = img.size

    records.append({
        "filename": filename,
        "id": Path(filename).stem,
        "width": width,
        "height": height,
        "aspect_ratio": width / height
    })

image_info = pd.DataFrame(records)
image_info = image_info.merge(df, on="id", how="left")

display(image_info.head())
print(image_info[["width", "height", "aspect_ratio"]].describe().round(2))

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(image_info["aspect_ratio"], bins=40)
plt.title("Aspect Ratio Distribution")
plt.xlabel("width / height")
plt.ylabel("count")
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(image_info["width"], image_info["height"], alpha=0.5)
plt.title("Image Width vs Height")
plt.xlabel("width")
plt.ylabel("height")
plt.show()

## 7. 밝기 분포 확인

In [ ]:
def mean_brightness(image_id):
    path = find_image_path(image_id)
    img = cv2.imread(str(path))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return float(gray.mean())

brightness_df = df.copy()
brightness_df["brightness"] = brightness_df["id"].apply(mean_brightness)

display(
    brightness_df.groupby("label")["brightness"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(2)
)

In [ ]:
plt.figure(figsize=(9, 5))

for label in sorted(brightness_df["label"].unique()):
    values = brightness_df.loc[
        brightness_df["label"] == label,
        "brightness"
    ]
    plt.hist(values, bins=40, alpha=0.5, label=f"label={label}", density=True)

plt.title("Brightness Distribution by Label")
plt.xlabel("Mean brightness")
plt.ylabel("Density")
plt.legend()
plt.show()

## 8. HSV 색상 분포 확인

In [ ]:
def mean_hsv(image_id):
    path = find_image_path(image_id)
    img = cv2.imread(str(path))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    return pd.Series({
        "h_mean": hsv[:, :, 0].mean(),
        "s_mean": hsv[:, :, 1].mean(),
        "v_mean": hsv[:, :, 2].mean()
    })

hsv_features = df["id"].apply(mean_hsv)
hsv_df = pd.concat([df, hsv_features], axis=1)

display(
    hsv_df.groupby("label")[["h_mean", "s_mean", "v_mean"]]
    .mean()
    .round(2)
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in zip(
    axes,
    ["h_mean", "s_mean", "v_mean"],
    ["Hue", "Saturation", "Value"]
):
    for label in sorted(hsv_df["label"].unique()):
        values = hsv_df.loc[hsv_df["label"] == label, col]
        ax.hist(values, bins=35, alpha=0.5, density=True, label=f"label={label}")

    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

## 9. 선명도 분포 확인

In [ ]:
def laplacian_variance(image_id):
    path = find_image_path(image_id)
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return float(cv2.Laplacian(img, cv2.CV_64F).var())

sharpness_df = df.copy()
sharpness_df["sharpness"] = sharpness_df["id"].apply(laplacian_variance)

display(
    sharpness_df.groupby("label")["sharpness"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)

In [ ]:
data_to_plot = [
    sharpness_df.loc[sharpness_df["label"] == 0, "sharpness"],
    sharpness_df.loc[sharpness_df["label"] == 1, "sharpness"]
]

plt.figure(figsize=(8, 5))
plt.boxplot(data_to_plot, tick_labels=["label 0", "label 1"], showfliers=True)
plt.title("Sharpness Distribution by Label")
plt.ylabel("Laplacian variance")
plt.show()

## 10. 극단값 이미지 확인

In [ ]:
def show_images_by_ids(ids, title, cols=5):
    rows = int(np.ceil(len(ids) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(16, 3.2 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, image_id in zip(axes, ids):
        img = Image.open(find_image_path(image_id)).convert("RGB")
        label = int(df.loc[df["id"] == image_id, "label"].iloc[0])

        ax.imshow(img)
        ax.set_title(f"{image_id}\nlabel={label}", fontsize=9)
        ax.axis("off")

    for ax in axes[len(ids):]:
        ax.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

darkest_ids = brightness_df.nsmallest(10, "brightness")["id"].tolist()
brightest_ids = brightness_df.nlargest(10, "brightness")["id"].tolist()
blurriest_ids = sharpness_df.nsmallest(10, "sharpness")["id"].tolist()

show_images_by_ids(darkest_ids, "Darkest Images")
show_images_by_ids(brightest_ids, "Brightest Images")
show_images_by_ids(blurriest_ids, "Lowest Sharpness Images")

## 11. 완전 중복 이미지 탐색

In [ ]:
import hashlib
from collections import defaultdict

def file_md5(path):
    hasher = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

hash_groups = defaultdict(list)

for filename in train_files:
    path = TRAIN_DIR / filename
    hash_groups[file_md5(path)].append(Path(filename).stem)

exact_duplicate_groups = [
    ids for ids in hash_groups.values()
    if len(ids) > 1
]

print("완전 중복 그룹 수:", len(exact_duplicate_groups))
print("완전 중복 이미지 수:", sum(len(group) for group in exact_duplicate_groups))

for group in exact_duplicate_groups[:10]:
    labels = df[df["id"].isin(group)][["id", "label"]]
    print("\n중복 그룹")
    display(labels)

## 12. EDA 후보 목록 작성

In [ ]:
candidate_columns = [
    "id",
    "label",
    "candidate_type",
    "reason",
    "review_status",
    "note"
]

candidate_df = pd.DataFrame(columns=candidate_columns)
candidate_df

후보 유형 예시:
- `possible_mislabel`
- `possible_duplicate`
- `possible_offtopic`
- `low_quality`
- `needs_review`

검토 결과 예시:
- `pending`
- `reviewed_keep`
- `reviewed_remove`
- `reviewed_relabel`

## 13. EDA 종합 결론

### 확인된 사실
- 전체 학습 이미지 수:
- 전체 테스트 이미지 수:
- 라벨 0 개수:
- 라벨 1 개수:
- 완전 중복 그룹 수:
- 이미지 크기 및 비율 특징:
- 밝기 분포 특징:
- 색상 분포 특징:
- 선명도 분포 특징:

### 의심되는 문제
- [ ] 완전 중복
- [ ] 근접 중복
- [ ] 라벨 기준 불일치
- [ ] off-topic 이미지
- [ ] 저품질 이미지
- [ ] 촬영 배경에 대한 shortcut learning
- [ ] 강제 Resize에 따른 왜곡 가능성

### 다음 단계
> 02_baseline.ipynb에서 원본 데이터를 사용한 기준 성능을 측정하고, 이후 데이터 정제 전후 ROC-AUC를 비교한다.